# Base API Tool
baseApiTool.py allows for easy implementation for AutoGen API tools.
This document will explain the purpose of each function and what can be changed for more advanced APIs.
To see this implemented, see adviceApiTool.ipynb

## Imports

In [1]:
from typing import Dict, Any, Type, Optional
from pydantic import BaseModel, Field
from langchain.tools import BaseTool
import requests

## API Input
This class is the input is required for the AutoGen agent to recognize the parameters for the tool.
It is also utilized for a dynamic parameter building process, which is shown in BaseAPITool.

Every API will require a baseUrl, which will access the API.

In this project, we avoided using paid APIs and limited APIs, so we did not have a key parameter but one can easily be added here.

In [2]:
class BaseAPIToolInput(BaseModel):
    baseUrl: str = Field(
        default=None,
        description="Base URL for the API endpoint."
    )

## BaseAPITool Class
This is the class where the logic for running the API based on the API input occurs.
The name, description, and args_schema are what allow for the agent to understand how a tool works.

### Build parameters
This function dynamically takes the parameters from BaseAPIToolInput and puts them in a way that the API will understand.

### Parse response
This function prepares the output of the API for the agent.

### Run
This function first builds the parameters based on the API Input. Next it calls the API and then returns the output, which is parsed in our parse_response function.

In [3]:
class BaseAPITool(BaseTool):
    name: str = "BaseAPITool"
    description: str = "Use this tool when you need to access an API."
    args_schema: Type[BaseModel] = BaseAPIToolInput

    def build_params(self, query: BaseModel) -> Dict[str, Any]:
        return {
            k: v for k, v in query.model_dump().items()
            if k != "baseUrl" and v is not None
        }

    def parse_response(self, response: requests.Response) -> Any:
        return response.json()

    def _run(self, **kwargs):
        try:
            query = self.args_schema(**kwargs)
            params = self.build_params(query)

            response = requests.get(query.baseUrl, params=params, timeout=10)            
            response.raise_for_status()
            return self.parse_response(response)
        except requests.exceptions.RequestException as e:
            return {"Error": str(e)}